# AI Agents LangGraph

## Why LangGraph?

LangChain Agents (AgentExecutor) were hard to customize and debug. LangGraph solves this by modeling agents as **stateful graphs**:

- **Nodes** = functions or LLM calls
- **Edges** = transitions between nodes (conditional or unconditional)
- **State** = typed dict shared across all nodes

This makes agent behavior explicit, debuggable, and composable.

---

## Core Concepts

```
StateGraph
│
├── State (TypedDict)
│     └── messages: list[BaseMessage]
│     └── custom_field: Any
│
├── Nodes (functions)
│     ├── agent_node(state) → state_update
│     └── tool_node(state) → state_update
│
└── Edges
      ├── START → agent_node
      ├── agent_node → tool_node (conditional)
      ├── agent_node → END (conditional)
      └── tool_node → agent_node
```

---

## State Management

State uses `Annotated` with reducers:
- `operator.add` for lists → appends instead of overwriting
- Custom reducers for conflict resolution

---

## Persistence / Checkpointing

LangGraph can checkpoint state after every node:
- **Resume** interrupted runs
- **Time-travel** to any past state
- **Human-in-the-loop** via `interrupt()`

Backends: `MemorySaver`, `SqliteSaver`, `PostgresSaver`

---

## Multi-Agent Patterns

### Supervisor Pattern
One orchestrator LLM routes tasks to specialized worker agents.

### Network Pattern  
Agents communicate peer-to-peer, passing messages via shared state.

### Hierarchical Pattern
Nested subgraphs each subgraph is itself a compiled graph used as a node.

In [1]:
# pip install langgraph langchain-openai langchain-core
import os
from typing import Annotated, Literal
import operator

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from typing_extensions import TypedDict

# ── Define State ──────────────────────────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]  # append-only

# ── Define Tools ──────────────────────────────────────────────────────────────
@tool
def search(query: str) -> str:
    """Search the web for information."""
    # Simulated search
    results = {
        "python": "Python is a high-level programming language.",
        "langchain": "LangChain is a framework for LLM applications.",
    }
    for key, val in results.items():
        if key in query.lower():
            return val
    return f"Search results for '{query}': Various relevant information found."

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"

tools = [search, calculator]

# ── LLM with Tools ────────────────────────────────────────────────────────────
llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY"))
llm_with_tools = llm.bind_tools(tools)

print("LangGraph setup complete")

LangGraph setup complete


In [2]:
# ── Build the Graph ───────────────────────────────────────────────────────────

# Node 1: Agent (LLM)
def agent_node(state: AgentState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# Node 2: Tools
tool_node = ToolNode(tools)

# Conditional edge: should we call tools or stop?
def should_continue(state: AgentState) -> Literal["tools", "__end__"]:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return "__end__"

# Build graph
graph = StateGraph(AgentState)
graph.add_node("agent", agent_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", should_continue)
graph.add_edge("tools", "agent")  # Always return to agent after tool

app = graph.compile()
print("Graph compiled")

# Run it
result = app.invoke({"messages": [HumanMessage(content="What is LangChain? Also calculate 42 * 17.")]})
print("\nFinal response:")
print(result["messages"][-1].content)

Graph compiled


In [3]:
# ── Persistence with MemorySaver ──────────────────────────────────────────────
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
app_with_memory = graph.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "user-123"}}

# First message
r1 = app_with_memory.invoke(
    {"messages": [HumanMessage(content="My name is Alex.")]},
    config=config
)
print("Turn 1:", r1["messages"][-1].content)

# Second message agent remembers previous context
r2 = app_with_memory.invoke(
    {"messages": [HumanMessage(content="What is my name?")]},
    config=config
)
print("Turn 2:", r2["messages"][-1].content)

In [4]:
# ── Human-in-the-Loop with interrupt ─────────────────────────────────────────
from langgraph.types import interrupt

def agent_with_approval(state: AgentState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    
    # If tool calls, ask for human approval
    if hasattr(response, "tool_calls") and response.tool_calls:
        tool_names = [tc["name"] for tc in response.tool_calls]
        # This pauses execution and waits for human input
        human_input = interrupt(f"Approve tool calls: {tool_names}? (yes/no)")
        if human_input.lower() != "yes":
            return {"messages": [AIMessage(content="Tool call cancelled by user.")]}
    
    return {"messages": [response]}

print("Human-in-the-loop pattern defined")
print("In production: app.invoke() returns with __interrupt__ state")
print("Resume with: app.invoke(Command(resume='yes'), config=config)")

Human-in-the-loop pattern defined
In production: app.invoke() returns with __interrupt__ state
Resume with: app.invoke(Command(resume='yes'), config=config)


In [5]:
# ── Streaming ─────────────────────────────────────────────────────────────────
for chunk in app.stream(
    {"messages": [HumanMessage(content="Calculate 100 * 200")]},
    stream_mode="updates"
):
    for node, update in chunk.items():
        print(f"Node '{node}': {str(update)[:100]}")

## Additional Learning Resources

### Documentation
- [LangGraph Docs](https://langchain-ai.github.io/langgraph/)
- [LangGraph Conceptual Guide](https://langchain-ai.github.io/langgraph/concepts/)
- [LangGraph Tutorials](https://langchain-ai.github.io/langgraph/tutorials/)
- [LangSmith (Observability)](https://docs.smith.langchain.com/)

### Videos
- [LangGraph Deep Dive LangChain YouTube](https://www.youtube.com/watch?v=quLe8HSzUVU)
- [Building Multi-Agent Systems with LangGraph](https://www.youtube.com/watch?v=hvAPnpSfSGo)

### Code
- [LangGraph Examples (GitHub)](https://github.com/langchain-ai/langgraph/tree/main/examples)
- [LangGraph Templates](https://github.com/langchain-ai/langgraph-template)